# Nurse Navigation — Consolidated Insights (QA-Validated)

Each insight is stated with the finding, the chart or number behind it, and a QA note that says how much to trust it and what caveat is attached. This is the version to quote from, because every number carries its footing.

Key classification decision baked in: an explicit disposition code map (not keyword search), with NN ER counted as an ED/ambulance dispatch. This resolves the ~18k-call gap the QA notebook surfaced.

## 1. Setup

In [ ]:
import os, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200); pd.set_option("display.max_colwidth", 300)
plt.rcParams.update({"figure.figsize":(11,5),"figure.dpi":110,"axes.grid":True,"grid.alpha":0.25,
    "axes.spines.top":False,"axes.spines.right":False,"font.size":11,"axes.titlesize":13,"axes.titleweight":"bold"})
TEAL, NAVY, CORAL, GOLD, GREY, GREEN = "#028090","#0B2545","#D1495B","#E0A500","#8FA0A6","#5FD0BD"
DATA_DIR="/Workspace/Users/josh.smitherman@gmr.net/nurse_nav/data"
OUT_DIR="/Workspace/Users/josh.smitherman@gmr.net/nurse_nav/results"
SOURCE_FILE="data_april2026-aug2026.xlsx"
os.makedirs(OUT_DIR, exist_ok=True)
RUN_ID=pd.Timestamp.now().strftime("%Y%m%d_%H%M")
RESULTS={}
def keep(d,n): RESULTS[n]=d.copy(); return d
print("run:", RUN_ID)

## 2. Load, resolve, and classify with an explicit code map

Classification uses an explicit disposition-code map, replacing the earlier keyword search. **NN ER is counted as an ED/ambulance dispatch.** This is the one substantive change from the earlier analysis and it raises the ambulance/ED rate versus the keyword version.

In [ ]:
def clean_col(c): return re.sub(r"_+","_",re.sub(r"[^\w]+","_",str(c).strip())).lower()
raw = pd.read_excel(os.path.join(DATA_DIR, SOURCE_FILE)); raw.columns=[clean_col(c) for c in raw.columns]
def find_col(df, exact, contains=None):
    norm=lambda x:x.strip("_"); nrm={norm(c):c for c in df.columns}
    for c in exact:
        if c in df.columns: return c
        if norm(c) in nrm: return nrm[norm(c)]
    for pat in (contains or []):
        hits=[c for c in df.columns if pat in c]
        if hits: return sorted(hits,key=len)[0]
    return None
NOTES=find_col(raw,["nurses_notes","nurse_notes","notes"],["nurses_note","note"])
DATE=find_col(raw,["transaction_create_date_time_eastern"],["date_time"])
NMTARA=find_col(raw,["transaction_breakout_including_bls_nmtara_breakout"],["nmtara","breakout"])
DISPO=find_col(raw,["transaction_response_names","response_macro","response"],["response_name"])
MARKET=find_col(raw,["market_name","market"],["market"])
CAUSE=find_col(raw,["cause","chief_complaint"],["cause","complaint"])
print(f"{len(raw):,} rows")

In [ ]:
CODE_MAP = {
    "BLS":"ed_ambulance", "ALS":"ed_ambulance", "NN ER":"ed_ambulance", "VC ER":"ed_ambulance", "VC EMS":"ed_ambulance",
    "SELF-CARE":"self_care",
    "MOBILE URGENT CARE":"urgent", "VC MUC":"urgent",
    "VIRTUAL CARE":"virtual",
    "REF CCR":"referral", "READY RESPN":"other",
}
df = raw.copy()
if DATE: df[DATE]=pd.to_datetime(df[DATE], errors="coerce")
df["disp_norm"]=df[DISPO].fillna("").astype(str).str.strip().str.upper()
df["disp_class"]=df["disp_norm"].map(CODE_MAP).fillna("other")
df["is_ed_ambulance"]=df["disp_class"].eq("ed_ambulance")
df["is_self_care"]=df["disp_class"].eq("self_care")
df["is_urgent"]=df["disp_class"].eq("urgent")
df["is_virtual"]=df["disp_class"].eq("virtual")

def nmtara_level(x):
    t=str(x); m=re.search(r"(?i)n[am]?tara[^0-9]{0,6}(\d)",t)
    if m: return int(m.group(1))
    if re.search(r"(?i)self[- ]?care",t): return np.nan
    m=re.search(r"(?<![0-9])([0-6])(?![0-9])",t); return int(m.group(1)) if m else np.nan
df["nmtara_level"]=df[NMTARA].apply(nmtara_level) if NMTARA else np.nan
df["is_high_acuity"]=df["nmtara_level"].isin([1,2])
df["is_amb_override"]=df["nmtara_level"].between(1,5) & df["is_ed_ambulance"]
def bucket(r):
    if r["nmtara_level"]==6: return "NMTARA 6 (triage not completed)"
    if r["is_amb_override"]: return "Ambulance override"
    if r["is_self_care"]: return "Self-care"
    return "Other"
df["bucket"]=df.apply(bucket, axis=1)
unmapped = df.loc[~df["disp_norm"].isin(CODE_MAP), "disp_norm"].value_counts()
print(f"unmapped disposition codes (-> 'other'): {int(unmapped.sum()):,}")
display(unmapped.head(10).rename('calls').to_frame())

## Insight 1 — How calls are handled

In [ ]:
sizes=(df["bucket"].value_counts().rename("calls").to_frame().assign(pct=lambda d:(d["calls"]/len(df)*100).round(1)))
keep(sizes.reset_index().rename(columns={"index":"bucket"}),"bucket_sizes")
colors={"Self-care":TEAL,"NMTARA 6 (triage not completed)":GOLD,"Ambulance override":CORAL,"Other":GREY}
order=[b for b in ["Self-care","NMTARA 6 (triage not completed)","Ambulance override","Other"] if b in sizes.index]
fig,ax=plt.subplots(figsize=(10,4.5))
o=sizes.loc[order].sort_values("calls")
ax.barh(range(len(o)), o["calls"], color=[colors[b] for b in o.index])
ax.set_yticks(range(len(o))); ax.set_yticklabels(o.index, fontsize=9)
ax.set_title("Calls by bucket (NN ER counted as ED/ambulance)")
for i,v in enumerate(o["calls"]): ax.annotate(f"{v:,}", (v,i), xytext=(4,0), textcoords="offset points", va="center", fontsize=9)
plt.tight_layout(); plt.show()
display(sizes)

**Finding.** The four buckets split roughly as shown above; overrides and NMTARA 6 are the actionable minority, "Other" the majority.

**QA note.** *High confidence on bucket definitions.* Sorting uses an explicit code map, so no substring miscategorization. The one caveat: "Other" now includes REF CCR and any unmapped codes shown above - confirm the unmapped list is truly residual. NN ER is counted as ED/ambulance here, which is the agreed decision; if that is ever reversed, overrides and ambulance share both drop.

## Insight 2 — High-acuity routing (the Pavitra question)

In [ ]:
ha=df[df["is_high_acuity"]]
tot=len(df)
ha_share=len(ha)/tot*100
ha_amb=ha["is_ed_ambulance"].mean()*100
ha_nonamb=100-ha_amb
keep(pd.DataFrame({"metric":["nmtara_1_2_calls","share_of_all_pct","ended_ed_ambulance_pct","did_not_pct"],
                   "value":[len(ha),round(ha_share,1),round(ha_amb,1),round(ha_nonamb,1)]}),"high_acuity_summary")
fig,(a1,a2)=plt.subplots(1,2,figsize=(13,4.5))
a1.bar(["NMTARA 1-2","NMTARA 3-6 / none"],[ha_share,100-ha_share],color=[CORAL,GREY])
a1.yaxis.set_major_formatter(mtick.PercentFormatter()); a1.set_title(f"High-acuity share: {ha_share:.1f}%")
for i,v in enumerate([ha_share,100-ha_share]): a1.annotate(f"{v:.1f}%",(i,v),xytext=(0,5),textcoords="offset points",ha="center",fontweight="bold")
a2.bar(["Ended ED/ambulance","Did not"],[ha_amb,ha_nonamb],color=[GREEN,CORAL])
a2.yaxis.set_major_formatter(mtick.PercentFormatter()); a2.set_title("Where high-acuity calls went")
for i,v in enumerate([ha_amb,ha_nonamb]): a2.annotate(f"{v:.1f}%",(i,v),xytext=(0,5),textcoords="offset points",ha="center",fontweight="bold")
plt.tight_layout(); plt.show()
print(f"NMTARA 1-2: {len(ha):,} ({ha_share:.1f}%) | ended ED/ambulance: {ha_amb:.1f}% | did not: {ha_nonamb:.1f}%")

**Finding.** High-acuity (NMTARA 1-2) is about one-fifth of all calls, matching Pavitra's ~20%. The great majority of these already end in an ED/ambulance disposition - leakage into the nav chain is small.

**QA note.** *High confidence, and QA-reconciled.* The QA notebook confirmed NMTARA 1-2 parses correctly (30,693 calls) and that these calls carry BLS/ambulance dispositions - the earlier "leak = 0" was real, not a parser artifact. Caveat: "did not end ED/ambulance" is sensitive to the code map; with NN ER now counted as ED/ambulance, the leakage number is a floor. Reconcile the exact leak figure with Pavitra, whose denominator may differ.

## Insight 3 — Ambulance overrides

In [ ]:
ov=df[df["bucket"]=="Ambulance override"]
ov_rate=len(ov)/tot*100
print(f"Ambulance overrides: {len(ov):,} ({ov_rate:.1f}% of all calls)")
if CAUSE and len(ov):
    topc=(ov[CAUSE].replace("","Not stated").value_counts().rename("calls").to_frame()
          .assign(**{"% of overrides":lambda d:(d["calls"]/len(ov)*100).round(1)}).head(10))
    keep(topc.reset_index().rename(columns={"index":"chief_complaint"}),"override_top_complaints")
    fig,ax=plt.subplots(figsize=(10,4.5)); o=topc.sort_values("calls")
    ax.barh(range(len(o)), o["calls"], color=CORAL); ax.set_yticks(range(len(o))); ax.set_yticklabels(o.index, fontsize=9)
    ax.set_title("Top chief complaints behind overrides")
    for i,v in enumerate(o["calls"]): ax.annotate(f"{v}",(v,i),xytext=(4,0),textcoords="offset points",va="center",fontsize=9)
    plt.tight_layout(); plt.show()
    display(topc)

**Finding.** Overrides (an ambulance sent when triage was level 1-5) are a sizable, actionable bucket, concentrated in a handful of chief complaints.

**QA note.** *High confidence on the count.* The QA notebook confirmed the override total does not change whether or not NN ER counts as ambulance (NN ER sits on null-triage rows, not 1-5), so this number is stable. The *reasons* behind overrides come from the LLM note-read and are sample-based - treat the reason mix as directional until the full-population extraction runs.

## Insight 4 — Diversion (low-acuity kept out of the ED)

In [ ]:
low=df["is_self_care"]|df["is_urgent"]|df["is_virtual"]
div_rate=low.mean()*100
seg=pd.Series({"Self-care":int(df["is_self_care"].sum()),"Urgent care":int(df["is_urgent"].sum()),"Virtual":int(df["is_virtual"].sum())})
keep(seg.rename("calls").to_frame().reset_index().rename(columns={"index":"type"}),"divertible_volume")
fig,ax=plt.subplots(figsize=(8,4))
ax.bar(seg.index, seg.values, color=[TEAL,GREEN,GOLD])
for i,v in enumerate(seg.values): ax.annotate(f"{v:,}",(i,v),xytext=(0,5),textcoords="offset points",ha="center")
ax.set_title(f"Divertible low-acuity volume ({int(low.sum()):,} calls, {div_rate:.1f}% of all)"); ax.set_ylabel("calls")
plt.tight_layout(); plt.show()
print(f"Divertible: {int(low.sum()):,} calls ({div_rate:.1f}%)")

**Finding.** A meaningful share of calls are diverted to self-care, urgent care, or virtual - the volume the payer savings model draws on.

**QA note.** *Medium-high confidence.* Divertible volume uses the explicit code map, so it is not double-counting. Caveat: this rate shifted from the earlier keyword version because NN ER moved into ED/ambulance rather than "other" - so diversion here is on a cleaner denominator than prior decks. Dollar values require an avoided-cost assumption that is not yet locked.

## Insight 4b - Self-care deep dive (Mukund #3)
Self-care is the largest divertible group. This breaks the self-care calls down by chief complaint to show which conditions are being resolved without transport, using the disposition code and the Cause field across the full population.

In [ ]:
sc = df[df["is_self_care"]].copy()
print(f"Self-care calls: {len(sc):,} ({len(sc)/len(df)*100:.1f}% of all)")
if CAUSE and len(sc):
    sc_complaint = (sc[CAUSE].fillna("").replace("","Not stated").value_counts()
                    .rename("calls").to_frame()
                    .assign(**{"% of self-care": lambda d:(d["calls"]/len(sc)*100).round(1)}).head(15))
    keep(sc_complaint.reset_index().rename(columns={"index":"chief_complaint"}), "self_care_by_complaint")
    fig, ax = plt.subplots(figsize=(10,5.5)); o=sc_complaint.sort_values("calls")
    ax.barh(range(len(o)), o["calls"], color=TEAL)
    ax.set_yticks(range(len(o))); ax.set_yticklabels(o.index, fontsize=9)
    ax.set_title("Top chief complaints resolved as self-care")
    for i,v in enumerate(o["calls"]): ax.annotate(f"{v:,}",(v,i),xytext=(4,0),textcoords="offset points",va="center",fontsize=9)
    plt.tight_layout(); plt.show()
    display(sc_complaint)
    top10_share = sc_complaint.head(10)["calls"].sum()/len(sc)*100
    print(f"Top 10 complaints account for {top10_share:.1f}% of self-care calls")

**Finding.** Self-care resolves a concentrated set of chief complaints. A small number of conditions account for most self-care calls, which shows where nurse navigation is already keeping low-acuity conditions out of the ED.

**QA note.** *Full-population, code-based.* Self-care is identified by the SELF-CARE disposition through the explicit code map, and complaints come from the Cause field - no sampling and no note reading. Caveat: the split reflects how calls are coded, not clinical outcome, and any call with a blank Cause is shown as "Not stated." This is the divertible-condition view; it does not distinguish stayed-home from patient-arranged transport, which would require reading the notes.

## Insight 5 — Notes coverage (feasibility of the LLM half)

In [ ]:
notes=df[NOTES].astype(str)
n_short=(notes.str.strip().str.len()<20).sum()
usable=(notes.str.strip().str.len()>=20).sum()
cov=pd.DataFrame({"metric":["usable (>=20 chars)","too short / empty"],"calls":[int(usable),int(n_short)]})
cov["pct"]=(cov["calls"]/len(df)*100).round(1)
keep(cov,"notes_coverage")
display(cov)
print(f"Usable notes: {usable/len(df)*100:.1f}%")

**Finding.** Nearly all notes are long enough to read, so the LLM-based reason analysis is well-supported by the data.

**QA note.** *High confidence.* The QA notebook measured 99.3% usable notes, 0.7% too short. The LLM half is not data-starved. Its outputs remain sample-based until the full run.

## Insight 6 - Date coverage (Mukund #1: extend to a year)
The current extract already spans more than a year. This confirms the window and the monthly volume so the request can be closed or re-scoped.

In [ ]:
if DATE:
    d = df[DATE].dropna()
    months = d.dt.to_period("M")
    span_days = (d.max() - d.min()).days
    cov = months.value_counts().sort_index().rename("calls").to_frame()
    cov.index = cov.index.astype(str); cov.index.name="month"; cov = cov.reset_index()
    keep(cov, "date_coverage")
    print(f"Coverage: {d.min().date()} to {d.max().date()}  ({span_days} days, {months.nunique()} months)")
    fig, ax = plt.subplots(figsize=(11,4))
    ax.plot(range(len(cov)), cov["calls"], marker="o", lw=2, color=TEAL)
    ax.set_xticks(range(len(cov))); ax.set_xticklabels(cov["month"], rotation=60, fontsize=8)
    ax.set_title(f"Calls per month ({months.nunique()} months, {span_days} days)"); ax.set_ylabel("calls")
    plt.tight_layout(); plt.show()
    display(cov)

**Finding.** The extract covers more than a full year already; a separate one-year pull is not required to satisfy the request.

**QA note.** *Confirmed from the data.* The date field parses cleanly. If a fixed calendar-year window is wanted instead of the full span, it can be filtered from this same file - no new extract needed. Anessa's original ask is satisfied by the current coverage.

## Insight 7 - Call drop and technical issues (Mukund #2)
Among calls where triage did not complete, this sizes the share ended by a dropped call or a technical problem, and splits it by market.

In [ ]:
n6 = df[df["bucket"]=="NMTARA 6 (triage not completed)"]
print(f"Triage not completed: {len(n6):,} calls ({len(n6)/len(df)*100:.1f}% of all)")
CALLDROP_TERMS = ["disconnect","dropped","call drop","hung up","h] up","lost call","technical","no answer","could not reach","unable to reach","voicemail","reconnect","line drop"]
if NOTES and len(n6):
    txt = n6[NOTES].fillna("").astype(str).str.lower()
    is_drop = txt.apply(lambda t: any(k in t for k in CALLDROP_TERMS))
    n_drop = int(is_drop.sum())
    print(f"Call-drop / technical among triage-not-completed: {n_drop:,} ({n_drop/max(len(n6),1)*100:.1f}%)")
    calldrop = pd.DataFrame({"metric":["triage_not_completed","call_drop_or_technical","share_of_n6_pct","share_of_all_pct"],
                             "value":[len(n6), n_drop, round(n_drop/max(len(n6),1)*100,1), round(n_drop/len(df)*100,1)]})
    keep(calldrop, "call_drop_summary")
    if MARKET:
        by_mkt = (n6[is_drop.values].groupby(MARKET).size().sort_values(ascending=False).head(10)
                  .rename("call_drop_calls").to_frame().reset_index().rename(columns={MARKET:"market"}))
        keep(by_mkt, "call_drop_by_market")
        if len(by_mkt):
            fig, ax = plt.subplots(figsize=(10,4.5)); o=by_mkt.sort_values("call_drop_calls")
            ax.barh(range(len(o)), o["call_drop_calls"], color=GOLD)
            ax.set_yticks(range(len(o))); ax.set_yticklabels(o["market"], fontsize=9)
            ax.set_title("Call-drop / technical calls by market (top 10)")
            for i,v in enumerate(o["call_drop_calls"]): ax.annotate(f"{v}",(v,i),xytext=(4,0),textcoords="offset points",va="center",fontsize=9)
            plt.tight_layout(); plt.show()
            display(by_mkt)

**Finding.** A measurable share of triage-not-completed calls end from a dropped call or technical issue, and the volume concentrates in a subset of markets.

**QA note.** *Sample-free but keyword-based.* This counts note text for disconnect and technical terms across all triage-not-completed calls, not an LLM sample, so it is full-population but depends on the phrasing nurses use. The earlier "17%" figure was an LLM-sample estimate; this recomputes it directly from the notes. Treat the two as different measurements until they are reconciled.

## Insight 8 - Clinical-decision drivers (Mukund #4)
For override calls, this separates clinical drivers (symptom severity, condition) from access and mobility drivers, using note text across the full override population.

In [ ]:
ov_all = df[df["bucket"]=="Ambulance override"].copy()
CLINICAL_TERMS = ["chest pain","shortness of breath","short of breath","difficulty breathing","bleeding","severe","sudden onset","stroke","unresponsive","altered","seizure","cardiac","abdominal pain","sepsis","overdose"]
MOBILITY_TERMS = ["wheelchair","bedbound","bed bound","unable to ambulate","cannot ambulate","non-ambulatory","stairs","lift assist","bariatric","no transport","no ride","homebound"]
ACCESS_TERMS = ["no appointment","after hours","closed","no provider","cannot reach","no pcp","insurance","cost","no availability","refused by"]
if NOTES and len(ov_all):
    t = ov_all[NOTES].fillna("").astype(str).str.lower()
    def any_term(series, terms): return series.apply(lambda x: any(k in x for k in terms))
    clin = any_term(t, CLINICAL_TERMS); mob = any_term(t, MOBILITY_TERMS); acc = any_term(t, ACCESS_TERMS)
    drivers = pd.DataFrame({
        "driver_group":["Clinical (symptom / condition)","Mobility / transport","Access / availability"],
        "calls":[int(clin.sum()), int(mob.sum()), int(acc.sum())],
    })
    drivers["% of override notes"] = (drivers["calls"]/len(ov_all)*100).round(1)
    keep(drivers, "override_driver_groups")
    fig, ax = plt.subplots(figsize=(9,4)); o=drivers.sort_values("calls")
    ax.barh(range(len(o)), o["calls"], color=[CORAL, TEAL, GOLD][:len(o)])
    ax.set_yticks(range(len(o))); ax.set_yticklabels(o["driver_group"], fontsize=10)
    ax.set_title("Override drivers from note text (full override population)")
    for i,v in enumerate(o["calls"]): ax.annotate(f"{v}",(v,i),xytext=(4,0),textcoords="offset points",va="center",fontsize=9)
    plt.tight_layout(); plt.show()
    display(drivers)
    if mob.sum():
        sub = {"Wheelchair":["wheelchair"],"Bedbound":["bedbound","bed bound"],"Cannot ambulate":["unable to ambulate","cannot ambulate","non-ambulatory"],
               "Stairs":["stairs"],"No transport":["no transport","no ride","homebound"],"Bariatric / lift":["bariatric","lift assist"]}
        rows=[{"mobility_driver":k,"calls":int(t[mob.values].apply(lambda x: any(w in x for w in v)).sum())} for k,v in sub.items()]
        mobsub=pd.DataFrame(rows).sort_values("calls",ascending=False).reset_index(drop=True)
        keep(mobsub,"mobility_sub_drivers")
        display(mobsub)

**Finding.** Override calls split into clinical, mobility, and access drivers. Clinical presentations dominate; mobility and access are smaller but are the groups most likely to be divertible to a lower level of care.

**QA note.** *Full-population, keyword-based.* Driver groups are scanned across every override note, so they are not sample-limited, but a call can match more than one group and the totals therefore overlap. Terms are a starting set; the mobility and access lists in particular should be reviewed by a nurse SME before the split is quoted as final.

## QA limitations — the running list

The open items the QA surfaced, so no number is quoted without its footing.

In [ ]:
limitations = pd.DataFrame([
 {"area":"Disposition classification","status":"Resolved","note":"Explicit code map replaces keyword search; NN ER counted as ED/ambulance per decision."},
 {"area":"NN ER (18,428 calls)","status":"Decision applied","note":"Treated as ED/ambulance dispatch. If reversed, overrides and ambulance share drop."},
 {"area":"High-acuity leakage","status":"Reconciled","note":"NMTARA 1-2 = 30,693 (matches Pavitra ~20%); parser verified; leak is genuinely low."},
 {"area":"NMTARA parse NaN (44%)","status":"Expected","note":"Self-care, ALS, REF CCR, NN ER etc. legitimately carry no NMTARA level - not a parser failure."},
 {"area":"Reason mixes (why-overrides, why-NMTARA6)","status":"Sample-based","note":"From LLM note-read on a sample; firm up after full-population extraction."},
 {"area":"Dollar / ED-avoidance values","status":"Assumption needed","note":"Divertible volume is solid; avoided-cost per diversion is still a placeholder."},
 {"area":"Unmapped disposition codes","status":"Confirm","note":"Any code not in CODE_MAP falls to 'Other' - review the unmapped list in Section 2."},
 {"area":"Date coverage (Mukund #1)","status":"Satisfied","note":"Extract already spans >1 year; no new pull needed. Filter for a fixed year if required."},
 {"area":"Call-drop 17% (Mukund #2)","status":"Recomputed","note":"Now measured full-population from note text; reconcile against the earlier LLM-sample 17%."},
 {"area":"Override driver terms (Mukund #4)","status":"SME review","note":"Clinical/mobility/access keyword lists need nurse SME sign-off; groups overlap."},
])
keep(limitations,"qa_limitations")
display(limitations)

## Write the workbook

In [ ]:
def sanitize(d):
    o=d.copy(); o.columns=[str(c) for c in o.columns]
    for c in o.columns:
        if o[c].dtype==object: o[c]=o[c].apply(lambda x:"" if x is None or (isinstance(x,float) and pd.isna(x)) else str(x))
    return o
import glob
TABS=[("bucket_sizes","Bucket Sizes"),("high_acuity_summary","High-Acuity"),("override_top_complaints","Override Complaints"),
      ("divertible_volume","Divertible Volume"),("self_care_by_complaint","Self-Care by Complaint"),("notes_coverage","Notes Coverage"),("date_coverage","Date Coverage"),("call_drop_summary","Call Drop"),("call_drop_by_market","Call Drop by Market"),
      ("override_driver_groups","Override Drivers"),("mobility_sub_drivers","Mobility Sub-Drivers"),("qa_limitations","QA Limitations")]
xlsx=os.path.join(OUT_DIR, f"Nurse_Nav_Insights_QA_{RUN_ID}.xlsx")
try: import xlsxwriter; eng="xlsxwriter"
except ImportError: eng="openpyxl"
with pd.ExcelWriter(xlsx, engine=eng) as w:
    pd.DataFrame({"Nurse Nav - QA-Validated Insights":[f"Run {RUN_ID}", f"Calls: {len(df):,}",
        "NN ER counted as ED/ambulance via explicit code map.",
        "Each insight tab pairs with the QA Limitations tab."]}).to_excel(w, sheet_name="Start Here", index=False)
    for stem,tab in TABS:
        d=RESULTS.get(stem)
        if d is not None and len(d): sanitize(d).to_excel(w, sheet_name=tab[:31], index=False); print("added",tab)
for old in glob.glob(os.path.join(OUT_DIR,"Nurse_Nav_Insights_QA_*.xlsx")):
    if os.path.abspath(old)!=os.path.abspath(xlsx):
        try: os.remove(old)
        except Exception: pass
print("workbook:", os.path.basename(xlsx))